In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection


In [ ]:
# cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection


In [ ]:
import numpy as np
BASE_DIR = "./dataset_windows20/"

X_train = np.load(BASE_DIR + "train/X.npy")
y_train = np.load(BASE_DIR + "train/y_bin.npy")

X_val = np.load(BASE_DIR + "val/X.npy")
y_val = np.load(BASE_DIR + "val/y_bin.npy")

X_test = np.load(BASE_DIR + "test/X.npy")
y_test = np.load(BASE_DIR + "test/y_bin.npy")


In [ ]:
X_train = X_train[:, :, 1:]
X_val   = X_val[:, :, 1:]
X_test  = X_test[:, :, 1:]

print(X_train.shape)


(279986, 20, 16)


In [ ]:
X_train = X_train[:, :, 0:-1]
X_val   = X_val[:, :, 0:-1]
X_test  = X_test[:, :, 0:-1]
print(X_train.shape)

(279986, 20, 15)


In [ ]:
## Preprocessing

In [ ]:
X_train_nom = X_train[y_train == 0]   # solo nominal

mean_feat = X_train_nom.mean(axis=(0,1))   # (F,)
std_feat  = X_train_nom.std(axis=(0,1)) + 1e-8


In [ ]:
def scale_windows(X, mean, std):
    return (X - mean[None, None, :]) / std[None, None, :]

X_train_s = scale_windows(X_train, mean_feat, std_feat)
X_val_s   = scale_windows(X_val,   mean_feat, std_feat)
X_test_s  = scale_windows(X_test,  mean_feat, std_feat)


In [ ]:
import pywt
import numpy as np

def compute_dwt_windows(X, wavelet="db4", level=1):
    """
    X: (N_windows, window_size, n_channels) o (N_windows, window_size)
    return:
        A: Approximation coefficients (low-frequency)
        D: Detail coefficients (high-frequency, concatenated)
    """
    if X.ndim == 2:
        X = X[..., np.newaxis]

    N, W, F = X.shape

    A_list = []
    D_list = []

    for i in range(N):
        A_ch = []
        D_ch = []

        for ch in range(F):
            coeffs = pywt.wavedec(X[i, :, ch], wavelet, level=level)

            A = coeffs[0]                  # Approximation
            D = np.concatenate(coeffs[1:]) # All detail levels

            A_ch.append(A)
            D_ch.append(D)

        A_list.append(np.stack(A_ch, axis=1))
        D_list.append(np.stack(D_ch, axis=1))

    return np.array(A_list), np.array(D_list)
# A_windows, D_windows = compute_dwt_windows(X)
# print("A:", A_windows.shape)
# print("D:", D_windows.shape)
def adaptive_threshold_detector_HF(window, thresholds):
    win_max = np.max(window, axis=0)
    return int(np.any(win_max > thresholds))
def adaptive_threshold_detector_LF(window, thresholds):
    win_min = np.min(window, axis=0)
    return int(np.any(win_min < thresholds))
def compute_thresholds(feature_windows, y_bin, k=2.0, mode="high"):
    # feature_windows: (N, coeffs, F)
    nominal = feature_windows[y_bin == 0]

    vals = np.max(nominal, axis=1) if mode == "high" else np.min(nominal, axis=1)

    mu = np.mean(vals, axis=0)
    sigma = np.std(vals, axis=0)

    if mode == "high":
        return mu + k * sigma
    else:
        return mu - k * sigma



In [ ]:
A_train, D_train = compute_dwt_windows(X_train_s)
A_val,   D_val   = compute_dwt_windows(X_val_s)
A_test,  D_test  = compute_dwt_windows(X_test_s)


In [ ]:
print(D_train.shape)
# (N_samples, n_coeffs, n_channels)


(279986, 13, 15)


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

N_train = D_train.shape[0]
N_val   = D_val.shape[0]
N_test  = D_test.shape[0]

D_train_flat = D_train.reshape(N_train, -1)
D_val_flat   = D_val.reshape(N_val, -1)
D_test_flat  = D_test.reshape(N_test, -1)

print("Original flattened shape:", D_train_flat.shape)


Original flattened shape: (279986, 195)


In [ ]:
mean = np.mean(D_train_flat, axis=0, keepdims=True)
std  = np.std(D_train_flat, axis=0, keepdims=True) + 1e-8

D_train_flat_n = (D_train_flat - mean) / std
D_val_flat_n   = (D_val_flat   - mean) / std
D_test_flat_n  = (D_test_flat  - mean) / std


In [ ]:
pca = PCA(n_components=0.95)

D_train_pca = pca.fit_transform(D_train_flat_n)
D_val_pca   = pca.transform(D_val_flat_n)
D_test_pca  = pca.transform(D_test_flat_n)

print("PCA shape:", D_train_pca.shape)


PCA shape: (279986, 119)


In [ ]:
explained = np.sum(pca.explained_variance_ratio_)
print("Total explained variance:", explained)


Total explained variance: 0.9516988959865246


In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([

    tf.keras.layers.Dense(
        16,
        activation='relu',
        input_shape=(D_train_pca.shape[1],)
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dense(8, activation='relu'),

    tf.keras.layers.Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    D_train_pca,
    y_train,
    validation_data=(D_val_pca, y_val),
    epochs=30,
    batch_size=64,
    verbose=1
)


Epoch 1/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.7649 - loss: 0.4787 - val_accuracy: 0.8862 - val_loss: 0.3406
Epoch 2/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.8643 - loss: 0.3433 - val_accuracy: 0.8948 - val_loss: 0.3450
Epoch 3/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.8670 - loss: 0.3362 - val_accuracy: 0.8990 - val_loss: 0.3453
Epoch 4/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.8701 - loss: 0.3315 - val_accuracy: 0.8973 - val_loss: 0.3257
Epoch 5/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.8695 - loss: 0.3281 - val_accuracy: 0.8992 - val_loss: 0.3331
Epoch 6/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.8714 - loss: 0.3239 - val_accuracy: 0.8994 - val_loss: 0.3293
Epoch 7/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.8726 - loss: 0.3214 - val_accuracy: 0.8997 - val_loss: 0.3369
Epoch 8/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.8738 - loss: 0

In [ ]:
y_pred = (model.predict(D_test_pca) > 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred, digits=4))


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
              precision    recall  f1-score   support

           0     0.8532    0.9728    0.9091     30406
           1     0.9673    0.8279    0.8922     29572

    accuracy                         0.9014     59978
   macro avg     0.9103    0.9004    0.9006     59978
weighted avg     0.9095    0.9014    0.9008     59978



# PCA no scaling

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

N_train = D_train.shape[0]
N_val   = D_val.shape[0]
N_test  = D_test.shape[0]

D_train_flat = D_train.reshape(N_train, -1)
D_val_flat   = D_val.reshape(N_val, -1)
D_test_flat  = D_test.reshape(N_test, -1)

print("Original flattened shape:", D_train_flat.shape)


Original flattened shape: (279986, 195)


In [ ]:
# mean = np.mean(D_train_flat, axis=0, keepdims=True)
# std  = np.std(D_train_flat, axis=0, keepdims=True) + 1e-8

# D_train_flat_n = (D_train_flat - mean) / std
# D_val_flat_n   = (D_val_flat   - mean) / std
# D_test_flat_n  = (D_test_flat  - mean) / std


In [ ]:
pca = PCA(n_components=0.95)

D_train_pca = pca.fit_transform(D_train_flat)
D_val_pca   = pca.transform(D_val_flat)
D_test_pca  = pca.transform(D_test_flat)

print("PCA shape:", D_train_pca.shape)


PCA shape: (279986, 89)


In [ ]:
explained = np.sum(pca.explained_variance_ratio_)
print("Total explained variance:", explained)


Total explained variance: 0.9503967834329684


In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([

    tf.keras.layers.Dense(
        16,
        activation='relu',
        input_shape=(D_train_pca.shape[1],)
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dense(8, activation='relu'),

    tf.keras.layers.Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    D_train_pca,
    y_train,
    validation_data=(D_val_pca, y_val),
    epochs=30,
    batch_size=64,
    verbose=1
)


Epoch 1/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.7171 - loss: 0.5408 - val_accuracy: 0.8181 - val_loss: 0.4158
Epoch 2/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 15s 3ms/step - accuracy: 0.8046 - loss: 0.4303 - val_accuracy: 0.8090 - val_loss: 0.4225
Epoch 3/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.8142 - loss: 0.4097 - val_accuracy: 0.8262 - val_loss: 0.3933
Epoch 4/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.8231 - loss: 0.3974 - val_accuracy: 0.8249 - val_loss: 0.3920
Epoch 5/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.8257 - loss: 0.3925 - val_accuracy: 0.8497 - val_loss: 0.3661
Epoch 6/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.8286 - loss: 0.3881 - val_accuracy: 0.8505 - val_loss: 0.3624
Epoch 7/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.8306 - loss: 0.3843 - val_accuracy: 0.8512 - val_loss: 0.3580
Epoch 8/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 32s 7ms/step - accuracy: 0.8322 - loss: 0

In [ ]:
y_pred = (model.predict(D_test_pca) > 0.5).astype(int)

from sklearn.metrics import classification_report, confusion_matrix
print(classification_report(y_test, y_pred, digits=4))


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
              precision    recall  f1-score   support

           0     0.8303    0.8752    0.8522     30406
           1     0.8641    0.8161    0.8394     29572

    accuracy                         0.8461     59978
   macro avg     0.8472    0.8456    0.8458     59978
weighted avg     0.8470    0.8461    0.8459     59978



The resulting tensors, originally structured as (N, n_coefficients, n_channels), are reshaped into flattened feature vectors per window to enable linear projection. Prior to dimensionality reduction, the data are standardized using the mean and standard deviation computed exclusively from the training set to ensure numerical stability and prevent data leakage. Principal Component Analysis (PCA) is then applied to the normalized training data to obtain an orthogonal low-dimensional representation that preserves the dominant variance structure of the detail coefficients. The same learned PCA transformation is subsequently applied to validation and test sets. The reduced feature vectors are finally fed into a lightweight fully connected neural network composed of two dense layers with ReLU activation and batch normalization, followed by a sigmoid output neuron for binary classification. This architecture combines linear variance-preserving compression with nonlinear decision boundaries, significantly reducing input dimensionality and computational complexity while maintaining discriminative power, making it particularly suitable for embedded deployment in resource-constrained onboard systems.